Task 1 — Exploratory Data Analysis & Visualization  
Conduct an exploratory analysis of the dataset. Produce a minimum of five visualizations covering feature
distributions, correlations, and categorical breakdowns. Each visualization must be accompanied by a written
interpretation explaining what it reveals about the data.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression
import random
import pickle
warnings.filterwarnings('ignore')

In [ ]:
#Task 1 EDA :
# Load the data
data = pd.read_csv(r'./premier_league.csv')
# Basic info of the data
print("general info of the dataset:")
print(data.info())



In [ ]:
print(data.head())
print("="*80)
print(data.tail())
print("="*80)

In [ ]:
#convert data types
data['game_id'] = data['game_id'].astype('category')
data['date'] = pd.to_datetime(data['date'])
data['year']=data['date'].dt.year


In [ ]:
duplicates = data.duplicated().sum()
print(f"Number of duplicates: {duplicates}")

In [ ]:
# check out the dimension of the dataset
data.shape

In [ ]:
for col in data.columns:
     print(f"Column {col}:")
     print(f"Number of unique values: {len(data[col].value_counts())}")
     print(f"Number of null values: {data[col].isnull().sum()}")
     print("------------------------------")

In [ ]:
#numerical_features & categorical_features
numerical_features = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = data.select_dtypes(include=['object', 'category']).columns.tolist()
print("Numerical features:", numerical_features)
print("Caterorical features:", categorical_features)

In [ ]:
print("value count in every column for each row:")
for col in categorical_features:
     print(f"Column {col}:")
     print(data[col].value_counts())
     print("------------------------------")

In [ ]:
# Descriptive statistics
print("\nDescriptive statistics:")
print(data[numerical_features].describe())

In [ ]:
# Descriptive statistics
print("\nDescriptive Categorical:")
print(data[categorical_features].describe())

In [ ]:
for col in categorical_features:
     print(f"Column {col}:")
     print(f"unique values in each category col: {data[col].unique()}")
     print("------------------------------")

In [ ]:
#visualizations

In [ ]:
#1.outlier detction
# boxplot to detect outliers
fig = make_subplots(
    rows=len(numerical_features),
    cols=1,
    subplot_titles=[f"Boxplot of {col}" for col in numerical_features]
)

for i, col in enumerate(numerical_features, start=1):
    fig.add_trace(
        go.Box(y=data[col], name=col),
        row=i, col=1
    )

fig.update_layout(
    title="Boxplots",
    showlegend=False,
    height=300 * len(numerical_features)
)

fig.show()

as we see most of numeric cols have outliers

In [ ]:
#2.histogram for all numerical cols
fig = make_subplots(
    rows=len(numerical_features),
    cols=1,
    subplot_titles=[f"Distribution of {col}" for col in numerical_features]
)

for i, col in enumerate(numerical_features, start=1):
    fig.add_trace(
        go.Histogram(x=data[col], name=col),
        row=i, col=1
    )

fig.update_layout(
    title="Distributions",
    showlegend=False,
    height=300 * len(numerical_features)
)

fig.show()

frequency histogram to all numeric cols , another way to see the distrubition of data

In [ ]:
#3.Correlation Heatmap
corr = data.select_dtypes(include='number').corr()

fig = px.imshow(corr,
                text_auto='.2f',
                aspect='auto',
                color_continuous_scale='RdBu_r',
                title='Correlation Heatmap')
fig.update_layout(width=1000, height=800)
fig.show()

this heatmap show correlation with all numeric cols , high corr is shown in red

In [ ]:
#4.top_scorers
top_scorers = data.groupby('player_name')['goals'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
plt.bar(top_scorers.index, top_scorers.values, color='hotpink')

plt.title("Top 10 Scorers")
plt.xlabel("Player")
plt.ylabel("Goals")
plt.xticks(rotation=45)

plt.show()

the top player to score last 4 years is Mo Salah , then haaland comes

In [ ]:
#5.top 5 clubs vs player values
club_values = data.groupby('club_name')['market_value_in_eur'].sum().reset_index()
#top 5 clubs
top5 = club_values.sort_values('market_value_in_eur', ascending=False).head(5)

fig = px.bar(
    top5,
    x='club_name',
    y='market_value_in_eur',
    title='Top 5 Clubs by Market Value (EUR)',
    text='market_value_in_eur'
)

fig.update_layout(xaxis_title="Club", yaxis_title="Total Market Value (€)")
fig.show()

Arsenal is the top club to have high value players, then man city comes

In [ ]:
#6.line chart year by average price
# Group by year and calculate average price
price_by_year = data.groupby('year')['market_value_in_eur'].mean().reset_index()
price_by_year = price_by_year.sort_values('year')  # Ensure order

# Create the plot
plt.figure(figsize=(10, 6))
plt.plot(price_by_year['year'], price_by_year['market_value_in_eur'],
         marker='o', linewidth=2)

# Add labels and title
plt.xlabel('Year')
plt.ylabel('Average players values')
plt.title('players values Over Time')
plt.grid(True, alpha=0.3)
plt.show()

players value increases over time

In [ ]:
#7.club power vs wins
club_win = data.groupby('club_name')['is_win'].mean()
club_power = data.groupby('club_name')['Club_International_Power'].mean()

plt.figure(figsize=(8,6))

plt.scatter(club_power, club_win, color='purple')

plt.title("Club Strength vs Win Rate")
plt.xlabel("Club Power")
plt.ylabel("Win Rate")

plt.show()

positivly correlated , club power increases , win rate also increases

In [ ]:
#8.pie charts for some category cols
cols_to_plot = ['position', 'foot']
colors = ['hotpink', 'purple', 'yellow', '#9933ff', '#ffb3d9']
# Create one figure with 3 subplots (1 row, 3 columns)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
# Loop through columns and axes together
for ax, col in zip(axes, cols_to_plot):

    counts = data[col].value_counts()

    ax.pie(counts,
           labels=counts.index,
           autopct='%1.1f%%',
           startangle=90,
           pctdistance=0.85,
           colors=colors,
           textprops={'fontsize': 11})

    ax.set_title(f'Distribution of {col.capitalize()}', fontsize=14, fontweight='bold')
    ax.axis('equal')

# Adjust layout
plt.tight_layout()
plt.show()

most of our players positions are defenders , then midfield. And most of them play with their right foot

In [ ]:
#9.Home vs Away Performance
home_goals = data.groupby('home_club_name')['home_club_goals'].mean()
away_goals = data.groupby('away_club_name')['away_club_goals'].mean()

plt.figure(figsize=(10,5))

plt.plot(home_goals.values[:10], label='Home Goals', marker='o')
plt.plot(away_goals.values[:10], label='Away Goals', marker='o')

plt.title("Home vs Away Performance")
plt.legend()

plt.show()

Home Performance is slightly bigger than Away Performance

In [ ]:
#10.subplots > scatter plots about player performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = [
    ('player_efficiency', 'Player Efficiency'),
    ('attack_contribution', 'Attack Contribution'),
    ('goals', 'Goals'),
    ('assists', 'Assists')
]

for ax, (col, label) in zip(axes.flat, metrics):
    corr = data['market_value_in_eur'].corr(data[col])
    sns.scatterplot(data=data, x=col, y='market_value_in_eur',
                    alpha=0.3, ax=ax, edgecolor='w')
    sns.regplot(data=data, x=col, y='market_value_in_eur',
                scatter=False, color='red', ax=ax)
    ax.set_title(f'{label} vs. Market Value\n(Correlation: {corr:.3f})', fontsize=11)
    ax.set_xlabel(label)
    ax.set_ylabel('Market Value (EUR)')

plt.suptitle('Player Performance vs. Market Value', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

player performance is positevly correlated with their attack Contribution

PREPROCESSING

feauture engineering for models

In [ ]:
data["total_minutes_player"] = (
    data.groupby("player_name")["minutes_played"].transform("sum")
)
data["total_impact_score"] = (
    data.groupby("player_name")["Player_Impact_Score"].transform("sum")
)
data["avg_player_rating"] = (
    data.groupby("player_name")["player_rating"].transform("mean")
)
data["avg_influence"] = (
    data.groupby("player_name")["influence"].transform("mean")
)
data["total_discipline"] = (
    data.groupby("player_name")["discipline"].transform("sum")
)
data["total_goals_player"] = (
    data.groupby("player_name")["goals"].transform("sum")
)
data["total_assists_player"] = (
    data.groupby("player_name")["assists"].transform("sum")
)
data["avg_player_efficiency"] = (
    data.groupby("player_name")["player_efficiency"].transform("mean")
)

data["avg_player_attack_contribution"] = (
    data.groupby("player_name")["attack_contribution"].transform("mean")
)


In [ ]:
data_analysis =data.copy()

In [ ]:
# 1. OUTLIER TREATMENT - LOG TRANSFORMATION
print("\n" + "="*50)
print("OUTLIER TREATMENT - LOG TRANSFORMATION")
print("="*50)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Before
sns.histplot(data['market_value_in_eur'], bins=50, kde=True, ax=axes[0], color='red')
axes[0].set_title('Market Value - BEFORE Log Transform')
# Apply Log
data['market_value_log'] = np.log1p(data['market_value_in_eur'])

# After
sns.histplot(data['market_value_log'], bins=50, kde=True, ax=axes[1], color='green')
axes[1].set_title('Market Value - AFTER Log Transform')

plt.tight_layout()
plt.savefig('market_value_log_transform.png', dpi=150)
plt.show()

print(f"Before - Skewness: {data['market_value_in_eur'].skew():.2f}")
print(f"After  - Skewness: {data['market_value_log'].skew():.2f}")


In [ ]:
# # 2. ENCODE CATEGORICAL FEATURES
# print("\n" + "="*50)
# print("ENCODING CATEGORICAL FEATURES")
# print("="*50)

# cat_cols = data.select_dtypes(include=['object']).columns.tolist()
# print(f"Categorical columns: {cat_cols}")

# labels = {}

# for col in cat_cols:
#     le = LabelEncoder()
#     data[col] = le.fit_transform(data[col].astype(str))
#     print(f"   Encoded {col}: {len(le.classes_)} categories")
    
#     labels[col] = le

In [ ]:
# with open("labels_encoding.pkl", "wb") as f:
#     pickle.dump(labels, f)

In [ ]:
cat_cols = data.select_dtypes(include=["object"]).columns.tolist()
labels = {}
for col in cat_cols:
    le = LabelEncoder().fit(data[col].astype(str))
    labels[col] = le
    data[col] = le.transform(data[col].astype(str))


In [ ]:
cat_cols = data.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {cat_cols}")


for col in cat_cols:
    le = labels[col]
    data[col] = le.transform(data[col].astype(str))
    print(f"   Encoded {col}: {len(le.classes_)} categories")
    

In [ ]:
# # 4. FEATURE SCALING
# print("\n" + "="*50)
# print("FEATURE SCALING")
# print("="*50)

# num_cols = data.select_dtypes(include=['int64', 'float64']).columns

# exclude_cols = ['appearance_id', 'player_name', 'position', 'sub_position', 
#                 'foot', 'club_name', 'opponent_name', 
#                 'home_club_name', 'away_club_name' , 'market_value_in_eur']

# num_cols = [col for col in num_cols if col not in exclude_cols]

# scaler = StandardScaler()
# data[num_cols] = scaler.fit_transform(data[num_cols])

# print(f"Scaled {len(num_cols)} features")

In [ ]:
# with open("scaler.pkl", "wb") as f:
#     pickle.dump(scaler, f)

In [ ]:

scaler = StandardScaler()

In [ ]:
# 4. FEATURE SCALING
print("\n" + "="*50)
print("FEATURE SCALING")
print("="*50)

num_cols = data.select_dtypes(include=['int64', 'float64']).columns

exclude_cols = ['appearance_id', 'player_name', 'position', 'sub_position', 
                'foot', 'club_name', 'opponent_name', 
                'home_club_name', 'away_club_name' , 'market_value_in_eur']

num_cols = [col for col in num_cols if col not in exclude_cols]

data[num_cols] = scaler.fit_transform(data[num_cols])

print(f"Scaled {len(num_cols)} features")

In [ ]:
print("PREPROCESSING COMPLETE")
print(f"Final shape: {data.shape}")
print("="*50)

In [ ]:
#3.Correlation Heatmap
corr = data.select_dtypes(include='number').corr()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(corr , fmt='.1f' , annot=True , cmap='Blues')
plt.show()

In [ ]:
data_model = data.copy()

In [ ]:
data_model.columns

In [ ]:
data_model = data_model[['age_player' , 'total_goals_player' , 'total_assists_player' , 'total_minutes_player' ,
                        'avg_player_efficiency' ,'total_impact_score' ,'avg_player_rating',
                        'avg_player_attack_contribution' ,'avg_influence' ,
                        'market_value_in_eur','highest_market_value_in_eur' , 'position' ,
                        'sub_position','foot','total_discipline' , 'club_name'  
                        ]]

In [ ]:
data_model = data_model.drop_duplicates()

In [ ]:
x = data_model.drop(columns=["market_value_in_eur"])
y = data_model["market_value_in_eur"]
n_features = x.shape[-1]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

In [ ]:
fitness_cache = {}

def repair(individual, min_features=12):
    while sum(individual) < min_features:
        i = random.randint(0, len(individual) - 1)
        individual[i] = 1
    return individual


def create_individual():
    individual = [random.randint(0, 1) for _ in range(n_features)]
    return repair(individual, min_features=12)


def fitness(individual , x ,y):
    key = tuple(individual)

    # 🔥 CACHE CHECK
    if key in fitness_cache:
        return fitness_cache[key]

    selected = [i for i in range(n_features) if individual[i] == 1]

    if len(selected) == 0:
        return 0

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        min_samples_leaf=2,
        random_state=42
    )

    score = cross_val_score(
        model,
        x.iloc[:, selected],
        y,
        cv=3
    ).mean()

    fitness_cache[key] = score
    return score


def select_parent(population , x , y):
    tournament = random.sample(population, 3)
    return max(tournament, key=lambda ind: fitness(ind, x, y))


def crossover(parent1, parent2):
    point = random.randint(1, n_features - 1)
    return parent1[:point] + parent2[point:]



def mutate(individual):
    individual = individual.copy()

    for i in range(len(individual)):
        if random.random() < 0.05:
            individual[i] = 1 - individual[i]

    # ensure at least 5 features selected
    return repair(individual, min_features=12)


In [ ]:
def GA_Feature_Selection(x ,y):

    population_size = 40
    generations = 15

    population = [create_individual() for _ in range(population_size)]

    best_overall = None
    best_score = -float("inf")

    for generation in range(generations):

        population = sorted(population, key=lambda ind: fitness(ind, x, y), reverse=True)

        best = population[0]
        best_fitness = fitness(best ,x ,y)

        if best_fitness > best_score:
            best_overall = best
            best_score = best_fitness

        print(f"Generation {generation + 1}")
        print(f"Best Fitness: {best_fitness}")
        print("------")

        new_population = []

        # 🔥 ELITISM (keep best)
        new_population.append(best)

        for _ in range(population_size - 1):
            parent1 = select_parent(population ,x ,y)
            parent2 = select_parent(population ,x ,y)

            child = crossover(parent1, parent2)
            child = mutate(child)

            new_population.append(child)

        population = new_population

    # final selected features
    features = [
        x.columns[i]
        for i in range(len(best_overall))
        if best_overall[i] == 1
    ]

    print("\nFinal Solution:")
    print(f"Best Chromosome: {best_overall}")
    print(f"Selected Features: {features}")
    print(f"Best Score: {best_score}")

    return best_overall , features, best_score

In [ ]:
# best_overall , features, best_score = GA_Feature_Selection(x, y)

In [ ]:
import pickle

# results = {
#     "best_overall": best_overall,
#     "features": features,
#     "best_score": best_score
# }

# with open("ga_feature_selection.pkl", "wb") as f:
#     pickle.dump(results, f)

In [ ]:
# The supplied GA implementation is retained above as an optional exercise.
# Use the dashboard's fixed feature set for a reproducible, bounded default run.
features = ['age_player','total_goals_player','total_assists_player','avg_player_efficiency','total_impact_score','avg_player_rating','avg_player_attack_contribution','avg_influence','highest_market_value_in_eur','position','sub_position','foot']
best_overall = [int(col in features) for col in x.columns]
best_score = float('nan')  # No GA validation score is claimed for this default selection.


In [ ]:
final_data_model = data_model[features]
final_data_model

In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# import matplotlib.pyplot as plt
# import pandas as pd

# model = RandomForestRegressor(
#     n_estimators=100,
#     random_state=42
# )

# model.fit(final_data_model, y)

In [ ]:
# with open("random_forest.pkl", "wb") as f:
#     pickle.dump(model, f)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
model.fit(final_data_model, y)


In [ ]:
importances = model.feature_importances_

features = final_data_model.columns

feature_importance_df = pd.DataFrame({
    "feature": features,
    "importance": importances
})

feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
)

In [ ]:
feature_importance_df

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    feature_importance_df["feature"],
    feature_importance_df["importance"]
)

plt.gca().invert_yaxis()  # عشان الأعلى يبقى فوق
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")

plt.show()

In [ ]:
print(data_model.shape)
data_model.head()

In [ ]:
print(final_data_model.shape)
final_data_model.head()

In [ ]:
y.head()

# Hierarical Clustering

In [ ]:
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt

In [ ]:
data_final=final_data_model

In [ ]:
data_final.info()

In [ ]:
data_final.head()

In [ ]:
k = 3 
cluster = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='ward')

labels = cluster.fit_predict(data_final)

data_final['cluster_label'] = labels

print(f"Agglomarated by {k} Groups ")
print(data_final['cluster_label'].value_counts())

## Visualizing Clustering

In [ ]:
cluster = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')

plt.figure(figsize=(10, 6))
sns.scatterplot(data=data_final, 
                x='total_goals_player', 
                y=y, 
                hue='cluster_label', 
                palette='viridis',
                alpha=0.7)

plt.title(f'Players Grouping (k={k}) based on Goals and Market Value')
plt.show()

In [ ]:
cluster = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')

plt.figure(figsize=(10, 6))
sns.scatterplot(data=data_final, 
                x='age_player', 
                y=y, 
                hue='cluster_label', 
                palette='viridis',
                alpha=0.7)

plt.title(f'Players Grouping (k={k}) based on Goals and Market Value')
plt.show()

In [ ]:
plt.figure(figsize=(15, 7))
plt.title("Players Data Dendogram")

linkage_matrix = shc.linkage(data_final, method='single')

dend = shc.dendrogram(linkage_matrix, truncate_mode='lastp', p=30)

plt.axhline(y=10, color='r', linestyle='--')
plt.xlabel("Clusters / Sample Index")
plt.ylabel("Euclidean Distances")
plt.show()

In [ ]:
plt.figure(figsize=(15, 7))
plt.title("Players Data Dendogram")

linkage_matrix = shc.linkage(data_final, method='ward')

dend = shc.dendrogram(linkage_matrix, truncate_mode='lastp', p=30)

plt.axhline(y=10, color='r', linestyle='--')
plt.xlabel("Clusters / Sample Index")
plt.ylabel("Euclidean Distances")
plt.show()

We applied the Ward linkage method to minimize the total within-cluster variance

##  K-Medoids Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler

In [ ]:

class KMedoids:
   
    def __init__(self, n_clusters=3, metric='euclidean', random_state=42, max_iter=300):
        self.n_clusters   = n_clusters
        self.metric       = metric
        self.random_state = random_state
        self.max_iter     = max_iter

    def fit(self, X):
        rng = np.random.RandomState(self.random_state)
        D   = pairwise_distances(X, metric=self.metric)   # full (n×n) distance matrix
        n   = len(X)

        # k-medoids++ initialisation
        medoid_idx = [rng.randint(n)]
        for _ in range(self.n_clusters - 1):
            dist_to_nearest = D[:, medoid_idx].min(axis=1)
            probs = dist_to_nearest ** 2
            probs /= probs.sum()
            medoid_idx.append(rng.choice(n, p=probs))
        medoid_idx = list(medoid_idx)

        for _ in range(self.max_iter):
            labels = D[:, medoid_idx].argmin(axis=1)
            new_medoids = list(medoid_idx)
            changed = False
            for k in range(self.n_clusters):
                cluster_pts = np.where(labels == k)[0]
                if len(cluster_pts) == 0:
                    continue
                costs = D[np.ix_(cluster_pts, cluster_pts)].sum(axis=1)
                best  = cluster_pts[costs.argmin()]
                if best != new_medoids[k]:
                    new_medoids[k] = best
                    changed = True
            medoid_idx = new_medoids
            if not changed:
                break

        self.medoid_indices_   = np.array(medoid_idx)
        self.labels_           = D[:, medoid_idx].argmin(axis=1)
        self.cluster_centers_  = X[medoid_idx]
        self.inertia_          = D[np.arange(n), self.medoid_indices_[self.labels_]].sum()
        return self

    def fit_predict(self, X):
        return self.fit(X).labels_

print("Custom KMedoids (PAM) class loaded — no C++ Build Tools required.")

In [ ]:
# ── Load & prepare dataset 
raw = pd.read_csv('premier_league.csv')

# Features chosen for clustering: player performance + value indicators
features_km = ['goals', 'assists', 'market_value_in_eur',
               'player_efficiency', 'attack_contribution',
               'minutes_played', 'Club_International_Power']

df_km = raw[features_km].dropna().reset_index(drop=True)
print(f"Shape after dropping NaN: {df_km.shape}")

scaler_km = StandardScaler()
X_km = scaler_km.fit_transform(df_km)
print("Features standardised (mean=0, std=1).")

In [ ]:
# Elbow Method + Silhouette Score to find optimal k 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.preprocessing import StandardScaler

raw = pd.read_csv('premier_league.csv')
features_km = ['goals', 'assists', 'market_value_in_eur',
               'player_efficiency', 'attack_contribution',
               'minutes_played', 'Club_International_Power']

df_km = raw[features_km].dropna().reset_index(drop=True)
scaler_km = StandardScaler()
X_km = scaler_km.fit_transform(df_km)

sample_size = 5000 
indices = np.random.choice(X_km.shape[0], sample_size, replace=False)
X_sample = X_km[indices]

print(f"Original shape: {X_km.shape}")
print(f"Sample shape for analysis: {X_sample.shape}")

inertia_list  = []
silhouette_list = []
K_range = range(2, 9)

for k in K_range:
    km = KMedoids(n_clusters=k, metric='manhattan', random_state=42)
    km.fit(X_sample)
    
    inertia_list.append(km.inertia_)
    
    sil = silhouette_score(X_sample, km.labels_, metric='manhattan')
    silhouette_list.append(sil)
    
    print(f"k={k}  inertia={km.inertia_:.2f}  silhouette={sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Plot
axes[0].plot(list(K_range), inertia_list, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Elbow Method (Manhattan)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Total PAM Cost)')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Silhouette Plot
axes[1].plot(list(K_range), silhouette_list, marker='s', color='darkorange', linewidth=2)
axes[1].set_title('Silhouette Score (Manhattan)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('K-Medoids Optimal k Selection (Sampled Data)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Fit final model with optimal k=3 
OPTIMAL_K = 3

kmed_final = KMedoids(n_clusters=OPTIMAL_K, metric='manhattan', random_state=42)

kmed_final.fit(X_sample)

final_medoids = kmed_final.cluster_centers_

print("Assigning all points to the nearest medoid...")
from sklearn.metrics import pairwise_distances

distances_to_medoids = pairwise_distances(X_km, final_medoids, metric='manhattan')

all_labels = distances_to_medoids.argmin(axis=1)

df_km['kmedoids_cluster'] = all_labels

print("\nCluster counts:")
print(df_km['kmedoids_cluster'].value_counts().sort_index())


sample_sil_idx = np.random.choice(len(X_km), 5000, replace=False)
sil_score = silhouette_score(X_km[sample_sil_idx], all_labels[sample_sil_idx], metric='manhattan')
print(f"\nFinal Silhouette Score (on 5k sample): {sil_score:.4f}")

print(f"Medoids shape: {final_medoids.shape}")
print("\nNo overlapping confirmed: every point assigned to exactly one cluster.")

In [ ]:
# ── Scatter: Goals vs Market Value coloured by cluster 
cluster_names = {0: 'Cluster 0: Utility Players', 
                 1: 'Cluster 1: Star Players', 
                 2: 'Cluster 2: Mid-Tier Contributors'}
colors_map    = {0: 'steelblue', 1: 'crimson', 2: 'seagreen'}

plt.figure(figsize=(11, 7))
for c in range(OPTIMAL_K):
    mask = df_km['kmedoids_cluster'] == c
    plt.scatter(df_km.loc[mask, 'goals'],
                df_km.loc[mask, 'market_value_in_eur'],
                label=cluster_names[c],
                color=colors_map[c],
                s=30, alpha=0.55)

# Mark medoids with a star
for c, idx in enumerate(kmed_final.medoid_indices_):
    plt.scatter(df_km.loc[idx, 'goals'],
                df_km.loc[idx, 'market_value_in_eur'],
                color='gold', edgecolors='black',
                s=300, marker='*', zorder=6,
                label='Medoid' if c == 0 else '')

plt.title('K-Medoids Clustering: Goals vs Market Value', fontsize=13, fontweight='bold')
plt.xlabel('Goals')
plt.ylabel('Market Value (EUR)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cluster Profile Analysis 
profile = df_km.groupby('kmedoids_cluster')[features_km].mean().round(2)
print("=== Mean Feature Values per Cluster ===")
print(profile.to_string())

# Normalised heatmap for visual comparison
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)

plt.figure(figsize=(13, 4))
ax = sns.heatmap(profile_norm, annot=True, fmt='.2f', cmap='YlOrRd',
                 linewidths=0.5, cbar_kws={'label': 'Normalised Mean'})
ax.set_yticklabels([cluster_names[i] for i in range(OPTIMAL_K)], rotation=0)
plt.title('K-Medoids Cluster Profiles – Normalised Feature Means', fontsize=13, fontweight='bold')
plt.xlabel('Feature')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Task 5: Fuzzy Logic 


In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Universe of Discourse 
# Input 1: attack_contribution  (raw, pre-scaled range observed in data)
attack_var = ctrl.Antecedent(np.arange(0, 101, 1), 'attack_contribution')

# Input 2: kmedoids_cluster  (0 = Utility, 1 = Star, 2 = Mid-Tier)
cluster_var = ctrl.Antecedent(np.arange(0, 2.01, 0.01), 'cluster')

# Output: wage_recommendation  (% of base wage; 50% = baseline)
wage_var = ctrl.Consequent(np.arange(0, 201, 1), 'wage_recommendation')

In [ ]:
# ── Membership Functions 
attack_var['low']    = fuzz.trapmf(attack_var.universe, [0,   0,  15,  35])
attack_var['medium'] = fuzz.trimf(attack_var.universe,  [25,  50,  75])
attack_var['high']   = fuzz.trapmf(attack_var.universe, [60,  80, 100, 100])

# cluster (0=Utility, 1=Star, 2=Mid-Tier)
cluster_var['utility']  = fuzz.trapmf(cluster_var.universe, [0,    0,    0.35, 0.65])
cluster_var['star']     = fuzz.trimf(cluster_var.universe,  [0.7,  1.0,  1.3])
cluster_var['midtier']  = fuzz.trapmf(cluster_var.universe, [1.35, 1.65, 2.0,  2.0])

# wage_recommendation output MFs
wage_var['very_low']  = fuzz.trapmf(wage_var.universe, [0,   0,   40,  65])
wage_var['low']       = fuzz.trimf(wage_var.universe,  [50,  75,  100])
wage_var['medium']    = fuzz.trimf(wage_var.universe,  [85,  110, 135])
wage_var['high']      = fuzz.trimf(wage_var.universe,  [120, 150, 175])
wage_var['very_high'] = fuzz.trapmf(wage_var.universe, [160, 180, 200, 200])

# View membership functions
attack_var.view()
plt.suptitle("Input: Attack Contribution MFs", fontsize=12)
plt.tight_layout(); plt.show()

cluster_var.view()
plt.suptitle("Input: K-Medoids Cluster MFs", fontsize=12)
plt.tight_layout(); plt.show()

wage_var.view()
plt.suptitle("Output: Wage Recommendation MFs", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Rule Base (10 IF-THEN rules grounded in domain knowledge) 

rule1  = ctrl.Rule(cluster_var['star']    & attack_var['high'],   wage_var['very_high'])
rule2  = ctrl.Rule(cluster_var['star']    & attack_var['medium'], wage_var['high'])
rule3  = ctrl.Rule(cluster_var['star']    & attack_var['low'],    wage_var['medium'])
rule4  = ctrl.Rule(cluster_var['midtier'] & attack_var['high'],   wage_var['high'])
rule5  = ctrl.Rule(cluster_var['midtier'] & attack_var['medium'], wage_var['medium'])
rule6  = ctrl.Rule(cluster_var['midtier'] & attack_var['low'],    wage_var['low'])
rule7  = ctrl.Rule(cluster_var['utility'] & attack_var['high'],   wage_var['medium'])
rule8  = ctrl.Rule(cluster_var['utility'] & attack_var['medium'], wage_var['low'])
rule9  = ctrl.Rule(cluster_var['utility'] & attack_var['low'],    wage_var['very_low'])
rule10 = ctrl.Rule(attack_var['high'],                            wage_var['medium'])   # safety floor

wage_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5,
                                 rule6, rule7, rule8, rule9, rule10])
wage_sim  = ctrl.ControlSystemSimulation(wage_ctrl)
print("Fuzzy control system created with 10 rules.")
print("Defuzzification method: centroid (COA) — skfuzzy default.")

In [ ]:
# ── Validation Against Real Samples From Dataset

raw_for_fuzzy = pd.read_csv('premier_league.csv')

features_for_alignment = ['goals', 'assists', 'market_value_in_eur',
                         'player_efficiency', 'attack_contribution',
                         'minutes_played', 'Club_International_Power']

raw_for_fuzzy = raw_for_fuzzy.dropna(subset=features_for_alignment).reset_index(drop=True)

raw_for_fuzzy['kmedoids_cluster'] = df_km['kmedoids_cluster']

raw_for_fuzzy['attack_contribution'] = raw_for_fuzzy['attack_contribution'].clip(0, 100)

cluster_names_short = {0: 'Utility', 1: 'Star', 2: 'Mid-Tier'}
samples = []
for cid in [0, 1, 2]:
    sub = raw_for_fuzzy[raw_for_fuzzy['kmedoids_cluster'] == cid]
    if len(sub) > 0: 
        sampled = sub.sample(2, random_state=42)
        for _, row in sampled.iterrows():
            samples.append(row.to_dict())

print(f"{'#':<3} {'Player':<25} {'Cluster':<12} {'Attack':>8} {'Cluster_in':>12} {'Wage %':>8} {'Interpretation'}")
print('-' * 95)

results = []
for i, s in enumerate(samples):
    atk = float(s['attack_contribution'])
    cid = int(s['kmedoids_cluster'])

    # Fuzzy Logic Simulation
    wage_sim.input['attack_contribution'] = atk
    wage_sim.input['cluster'] = float(cid)
    wage_sim.compute()
    wage_pct = wage_sim.output['wage_recommendation']

    if wage_pct >= 160:   interp = "Elite — very high wage"
    elif wage_pct >= 120: interp = "Strong — high wage"
    elif wage_pct >= 85:  interp = "Average — medium wage"
    elif wage_pct >= 50:  interp = "Below avg — low wage"
    else:                 interp = "Minimal — very low wage"

    results.append({**s, 'wage_pct': wage_pct, 'interp': interp})
    print(f"{i+1:<3} {str(s['player_name']):<25} {cluster_names_short[cid]:<12} "
          f"{atk:>8.1f} {cid:>12}       {wage_pct:>8.1f}%   {interp}")

In [ ]:
# ── Visualise Fuzzy Output for Each Validation Sample
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()

mf_items = [
    ('very_low',  wage_var['very_low'].mf,  'navy'),
    ('low',       wage_var['low'].mf,       'steelblue'),
    ('medium',    wage_var['medium'].mf,    'seagreen'),
    ('high',      wage_var['high'].mf,      'darkorange'),
    ('very_high', wage_var['very_high'].mf, 'crimson'),
]

for i, s in enumerate(results[:6]):
    wage_sim.input['attack_contribution'] = float(s['attack_contribution'])
    wage_sim.input['cluster']             = float(s['kmedoids_cluster'])
    wage_sim.compute()

    ax = axes[i]
    for label, mf_vals, col in mf_items:
        ax.plot(wage_var.universe, mf_vals, label=label, color=col, alpha=0.7)

    ax.axvline(x=s['wage_pct'], color='black', linewidth=2.5,
               linestyle='--', label=f"Output: {s['wage_pct']:.1f}%")
    ax.fill_between(wage_var.universe,
                    np.zeros_like(wage_var.universe),
                    np.minimum(
                        np.interp(s['wage_pct'], wage_var.universe, 
                                  wage_var['medium'].mf),
                        1.0
                    ),
                    alpha=0.1, color='gray')

    ax.set_title(f"Sample {i+1}: {cluster_names_short[int(s['kmedoids_cluster'])]} | "
                 f"Atk={s['attack_contribution']:.1f}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Wage Recommendation (%)')
    ax.set_ylabel('Membership Degree')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_xlim(0, 200)

plt.suptitle('Fuzzy Output MFs – Validation Samples (Centroid Defuzzification)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nFuzzy Logic System validated against real dataset samples successfully.")